# Feature Screening: Rank IC on 10 Years of EUR/USD 15-Minute Bars

Measures whether simple technical features predict the next 1, 4, and 16 bars of returns, using Spearman rank correlation (rank IC). Also checks how much the features overlap with each other, and whether the strongest signal (RSI) keeps the same sign across three market periods.

**Key results:** RSI(14) has a rank IC of -0.036 against the next bar (a mean-reversion sign), and the sign holds in 2016-2019, 2020-2022, and 2023-2026. RSI and EMA distance are 0.92 correlated, so they carry largely the same information. Realized volatility shows no predictive power.

*Data (broker 15-minute bars) is not included in this repository.*

In [1]:
import pandas as pd

df = pd.read_csv("../Data/FixTimeBars/EURUSD_15M_Demo_Ready.csv")
print(df.shape)
print(df.columns.tolist())
df.head()

(248497, 10)
['time', 'open', 'high', 'low', 'close', 'tick_volume', 'spread', 'real_volume', 'low_time', 'high_time']


,time,open,high,low,close,tick_volume,spread,real_volume,low_time,high_time
0,2016-10-13 00:00:00,1.10063,1.10110,1.10036,1.10063,455,0,223012000,2016-10-13 00:00:00,2016-10-13 00:07:00
1,2016-10-13 00:15:00,1.10062,1.10102,1.10062,1.10090,430,11,255276000,2016-10-13 00:15:00,2016-10-13 00:21:00
2,2016-10-13 00:30:00,1.10091,1.10115,1.10081,1.10108,284,9,242116000,2016-10-13 00:30:00,2016-10-13 00:42:00
3,2016-10-13 00:45:00,1.10107,1.10111,1.10058,1.10079,356,9,212271000,2016-10-13 00:51:00,2016-10-13 00:45:00
4,2016-10-13 01:00:00,1.10079,1.10086,1.10049,1.10067,638,7,567767000,2016-10-13 01:14:00,2016-10-13 01:11:00


In [2]:
df["time"] = pd.to_datetime(df["time"], format="%Y-%m-%d %H:%M:%S")
df = df.set_index("time")

print(df.shape)
print(df.index.min(), "to", df.index.max())

(248497, 9)
2016-10-13 00:00:00 to 2026-10-13 23:45:00


In [4]:
import numpy as np

df["log_return"] = np.log(df["close"]).diff()
df["future_log_return"] = df["log_return"].shift(-1)

df[["close", "log_return", "future_log_return"]].head()

,close,log_return,future_log_return
time,,,
2016-10-13 00:00:00,1.10063,NaN,0.000245
2016-10-13 00:15:00,1.10090,0.000245,0.000163
2016-10-13 00:30:00,1.10108,0.000163,-0.000263
2016-10-13 00:45:00,1.10079,-0.000263,-0.000109
2016-10-13 01:00:00,1.10067,-0.000109,0.000327


In [5]:
for k in [1, 4, 16]:
    df[f"fwd_{k}"] = (np.log(df["close"]).shift(-k) - np.log(df["close"])) * 10000

print(df[["fwd_1", "fwd_4", "fwd_16"]].describe().round(2))

           fwd_1      fwd_4     fwd_16
count  248496.00  248493.00  248481.00
mean        0.00       0.00       0.01
std         4.81       9.41      18.64
min      -114.65    -163.19    -206.89
25%        -1.98      -3.96      -8.27
50%         0.00       0.00       0.09
75%         1.98       3.96       8.34
max       162.29     185.15     252.03


In [6]:
import ta

df["rsi_14"] = ta.momentum.RSIIndicator(close=df["close"], window=14).rsi()

ema_50 = ta.trend.EMAIndicator(close=df["close"], window=50).ema_indicator()
df["ema_dist"] = (df["close"] - ema_50) / ema_50 * 10000

df["vol_20"] = df["close"].pct_change().rolling(20).std() * 10000

print(df[["rsi_14", "ema_dist", "vol_20"]].describe().round(2))

          rsi_14   ema_dist     vol_20
count  248484.00  248448.00  248477.00
mean       50.06       0.02       4.07
std        11.71      16.00       2.57
min         7.01    -137.57       0.49
25%        42.13      -7.83       2.36
50%        50.09      -0.04       3.47
75%        57.97       7.79       5.01
max        93.35     178.47      38.31


In [7]:
from scipy.stats import spearmanr

feats = ["rsi_14", "ema_dist", "vol_20"]
clean = df.dropna(subset=feats + ["fwd_1", "fwd_4", "fwd_16"])

for f in feats:
    row = []
    for k in [1, 4, 16]:
        rho, p = spearmanr(clean[f], clean[f"fwd_{k}"])
        row.append(f"fwd_{k}: rho={rho:+.4f} p={p:.4f}")
    print(f"{f:10s}  " + "   ".join(row))

print()
print(clean[feats].corr(method="spearman").round(3))

rsi_14      fwd_1: rho=-0.0355 p=0.0000   fwd_4: rho=-0.0278 p=0.0000   fwd_16: rho=-0.0171 p=0.0000
ema_dist    fwd_1: rho=-0.0293 p=0.0000   fwd_4: rho=-0.0233 p=0.0000   fwd_16: rho=-0.0180 p=0.0000
vol_20      fwd_1: rho=-0.0017 p=0.3996   fwd_4: rho=-0.0041 p=0.0410   fwd_16: rho=+0.0012 p=0.5573

          rsi_14  ema_dist  vol_20
rsi_14     1.000     0.916  -0.012
ema_dist   0.916     1.000  -0.006
vol_20    -0.012    -0.006   1.000


In [8]:
yr = clean.index.year
for period in [(2016, 2019), (2020, 2022), (2023, 2026)]:
    sub = clean[(yr >= period[0]) & (yr <= period[1])]
    rho, p = spearmanr(sub["rsi_14"], sub["fwd_1"])
    print(f"{period[0]}-{period[1]}:  n={len(sub):6d}  rho={rho:+.4f}  p={p:.4f}")

2016-2019:  n= 79787  rho=-0.0420  p=0.0000
2020-2022:  n= 74727  rho=-0.0354  p=0.0000
2023-2026:  n= 93918  rho=-0.0301  p=0.0000
